In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
# tool create

@tool
def multiply(a: int, b: int) -> int:
  """Given 2 numbers a and b this tool returns their product"""
  return a * b

In [3]:
print(multiply.invoke({'a':3, 'b':4}))

12


In [4]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [5]:
llm.invoke('hi')

AIMessage(content='Hi there! How can I help you today?', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f241f-b0f3-7c51-9578-de3c7ebffa06-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 2, 'output_tokens': 347, 'total_tokens': 349, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 337}})

In [6]:
llm_with_tools = llm.bind_tools([multiply])

In [7]:
llm_with_tools.invoke('Hi how are you')

AIMessage(content="I'm doing well, thank you! How can I help you today?", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f2420-f180-7862-9ffb-2cea2f0d83ae-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 58, 'output_tokens': 16, 'total_tokens': 74, 'input_token_details': {'cache_read': 0}})

In [8]:
query = HumanMessage('can you multiply 3 with 1000')

In [10]:
messages = [query]
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={})]

In [11]:
result = llm_with_tools.invoke(messages)
messages.append(result)

In [12]:
messages

[HumanMessage(content='can you multiply 3 with 1000', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'multiply', 'arguments': '{"a": 3, "b": 1000}'}, '__gemini_function_call_thought_signatures__': {'e435cbf1-24a2-4c0b-a613-10854ddbfdfe': 'CpUCAQw51scGUbkJWx/aCQxiaE9mxWaDgd/IOCWE+5rUvxHOmFqZ2kPTffnYJIZuxzoAv9rhljz00Q2hbLy9y109VyC9drSxM5n+/QvDfPdlJyZbiBItsd1cZ/xJLv6Xn8FpQdcZ2XY2wIVOVsshu2ueyxrX+zan3fUrB5kKZz7BMtfAXlcThH32VazURk2+HyDBACWbnG4IhgqcEO4VmlR3UvK1ZBC44Ga6skH22kUjYGq1n8eZVJiPK33WYA+iX+gM81EN/29YdnRPDrN0edHbvvIEgiRoVIQRd3fNHj5SbUAd5AhtTMxbYDeXq5hdvhfsU3vvJGuFq6sskGPta3kQE4WKfYTCJQpxoAdf1hCBfbq/rdx3hA=='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f2421-c79d-7393-b266-bc12ff5d6b39-0', tool_calls=[{'name': 'multiply', 'args': {'a': 3, 'b': 1000}, 'id': 'e435cbf1-24a2-4c0b-a613-10854ddbfdfe', 'type': 'tool_call

In [13]:
tool_result = multiply.invoke(result.tool_calls[0])

In [14]:
tool_result

ToolMessage(content='3000', name='multiply', tool_call_id='e435cbf1-24a2-4c0b-a613-10854ddbfdfe')

In [15]:
llm_with_tools.invoke(messages).content

''

In [16]:
# tool create
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_currency: str, target_currency: str) -> float:
  """
  This function fetches the currency conversion factor between a given base currency and a target currency
  """
  url = f'https://v6.exchangerate-api.com/v6/c754eab14ffab33112e380ca/pair/{base_currency}/{target_currency}'

  response = requests.get(url)

  return response.json()

@tool
def convert(base_currency_value: int, conversion_rate: Annotated[float, InjectedToolArg]) -> float:
  """
  given a currency conversion rate this function calculates the target currency value from a given base currency value
  """

  return base_currency_value * conversion_rate


In [17]:
get_conversion_factor.invoke({'base_currency':'USD','target_currency':'INR'})

{'result': 'success',
 'documentation': 'https://www.exchangerate-api.com/docs',
 'terms_of_use': 'https://www.exchangerate-api.com/terms',
 'time_last_update_unix': 1782950402,
 'time_last_update_utc': 'Thu, 02 Jul 2026 00:00:02 +0000',
 'time_next_update_unix': 1783036802,
 'time_next_update_utc': 'Fri, 03 Jul 2026 00:00:02 +0000',
 'base_code': 'USD',
 'target_code': 'INR',
 'conversion_rate': 95.2406}

In [18]:
convert.invoke({'base_currency_value':10, 'conversion_rate':85.16})

851.5999999999999

In [19]:
llm_with_tools = llm.bind_tools([get_conversion_factor, convert])

In [20]:
messages = [HumanMessage('What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd')]
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={})]

In [21]:
ai_message = llm_with_tools.invoke(messages)
messages.append(ai_message)

In [22]:
ai_message.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'target_currency': 'USD', 'base_currency': 'INR'},
  'id': '8ad84f25-4322-4b80-afb0-cd6006e779ec',
  'type': 'tool_call'}]

In [23]:
import json

for tool_call in ai_message.tool_calls:
  # execute the 1st tool and get the value of conversion rate
  if tool_call['name'] == 'get_conversion_factor':
    tool_message1 = get_conversion_factor.invoke(tool_call)
    # fetch this conversion rate
    conversion_rate = json.loads(tool_message1.content)['conversion_rate']
    # append this tool message to messages list
    messages.append(tool_message1)
  # execute the 2nd tool using the conversion rate from tool 1
  if tool_call['name'] == 'convert':
    # fetch the current arg
    tool_call['args']['conversion_rate'] = conversion_rate
    tool_message2 = convert.invoke(tool_call)
    messages.append(tool_message2)



In [24]:
messages

[HumanMessage(content='What is the conversion factor between INR and USD, and based on that can you convert 10 inr to usd', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'function_call': {'name': 'get_conversion_factor', 'arguments': '{"target_currency": "USD", "base_currency": "INR"}'}, '__gemini_function_call_thought_signatures__': {'8ad84f25-4322-4b80-afb0-cd6006e779ec': 'CrgMAQw51sc9hpfAE2rMg2K//VZmGMYsxKvGPMcMr0mXzEL54klqx0axmscI2RXZWWf/v92PZ27WjkQkjD6l9rdoW4Jh0ZB1EZR2UBYHr1fclI0RpgO1q+Nc05DzYozqELH6hMvgg4CuVTwPoyNwmdA1MgcUxxsz9pPm9SZiecSOnhNXGdRlHlzj8q54qqPg3IcZ0VYcNK8gYZrvmtyHE0i46rSYwtESk9qrGiYzwGFNLJl8lW4pTwjrpK76Qc0c8H34dUCmmbOZ4SxxT7/nrpmJO5eVsDamolqiwdvH/35UBh4HqcWO/v1sqLXmrM4MBNeobveg1Y03j2QdbyilUUETu2h0jMbcV9dSYb3B7tvhLx9YIRuwVxyJatjEapq7ffFZ27D3845ZXRgNdNc5N+nwzxfveBdUxusQQHiOTU5hrComJFAb+IDkbSLW0lC8CgDy2fNTodK4ReEBxYX+uAQDd3kkpyBhqqMhyLObXzzzL26LNRr6ZPmUc5bLVhMu9GZlbwReTkteZ0siffn2EeaaFxlRiJnbTKNDYDLYpbtH/RsltJAGrxaWWsQdd78DrBD8V

In [25]:
llm_with_tools.invoke(messages).content

[{'type': 'text',
  'text': 'The conversion factor between INR and USD is 0.0105.\n\nI cannot directly convert 10 INR to USD using the available `convert` function, as it does not allow for specifying the target currency or the conversion rate obtained from `get_conversion_factor`.',
  'extras': {'signature': 'CoELAQw51sec2QS98IvWsEcvo6d9X+NyWKQhlswrtORqvV86a0m7c6OJR/1xiLEQblbI2lUsIHd4QQxORgjY2tqU39wuUe4W1ckqOFVh1ayv+iTCmBPtIFRraR4otig04y1kG7Vc7MN6iVLzy+FCyBnZjIMOT5TgnhDiinddelx34W+/xN8/YaiaOBvdkC/3Xcv3g3RqVqNN2eMgLuILUd4jd7CG7mSlYBq+Tim2WqdZ71LYdFZYI2zd8m6a3H5aftXYIrUy9OEAJOcdYMTwWt7kDaG+7x9fckuB5Tyy7Fj07ADOkZ3hMFIoIvrsgnHtoPBUqwlW3VPwhemDAGZ/1KfF1gpGWAE4+Cw97u0Cr+nSuNmUpSFg+cRctZzl4VwyoUJQjZcauk57p6yDw1Bd1oWYxvQAq6pL9iQ7rJ5E6J645fqi3/oRrrilDG1UtTpfrfyxXlXMEZJSOdYY9YD4sg9+GkiXHfFfyVreFmD71P8UY/zhnsN24qyu4FOYWWMy3qaci8t+/to9b9hqziFj2L3tyFqymPNqnqUVY09Uc8OEGafMPFB6Wlf8Qsf+xj4i7iduNy/nKV7VKVbqR7aUC00PgMaXjVnz4WGTyAIWfBARH8iATfOK+grpWyGYg+ZxdgIM7dtQDOXLtydDYQLc3wb/g5KNVfhxyPhKJyJyQ+NM5Udn

In [27]:
from langchain_classic.agents import initialize_agent, AgentType

# Step 5: Initialize the Agent ---
agent_executor = initialize_agent(
    tools=[get_conversion_factor, convert],
    llm=llm,
    agent=AgentType.STRUCTURED_CHAT_ZERO_SHOT_REACT_DESCRIPTION,  # using ReAct pattern
    verbose=True  # shows internal thinking
)
# --- Step 6: Run the Agent ---
user_query = "Hi how are you?"

response = agent_executor.invoke({"input": user_query})

C:\Users\ishua\AppData\Local\Temp\ipykernel_21308\2550726259.py:4: LangChainDeprecationWarning: Use `langchain.agents.create_agent` for new applications. It provides a more flexible agent factory with middleware support, structured output, and integration with LangGraph for persistence, streaming, and human-in-the-loop workflows. Migration guide: https://docs.langchain.com/oss/python/migrate/langchain-v1
  agent_executor = initialize_agent(




> Entering new AgentExecutor chain...
Thought: The user is asking a social question, not one that requires tool use. I should respond directly.
Action:
```
{
  "action": "Final Answer",
  "action_input": "I'm doing well, thank you! How can I help you today?"
}
```

> Finished chain.
